# 第三題：凍結後的事後分析（Qwen3-14B）

這不是凍結系統的一部分。計畫在執行前寫好：`evaluation/posthoc14b-plan.md`（sha256 `90e52e6443cabe7503fd7420624a09109bb6bb64f2351bb511ad0382deb4fd69`）。結果不論好壞，都不用來調整 prompt、知識卡、模型或微調權重，也不改寫既有判定。

**要回答的問題**：YMYL 依主題過度標記，是 Qwen3-8B 能力不足，還是 prompt 或任務框架的問題。同一個凍結 prompt 換成 Qwen3-14B（revision `40c06982…`），用和凍結系統相同的 4-bit 設定載入單張 T4、不載微調權重，審同樣 8 篇測試文章，和既有的 8B 4-bit 無微調輸出比較。

載入、組 prompt、greedy 生成、最多一次重寫與輸出契約都從凍結 bundle 匯入、不修改；只把凍結的 `load()` 讀到的模型名稱與版本換成 14B。讀測試資料之前，先檢查磁碟、chat template 與 fp16 logits。

In [ ]:
import os,sys,json,hashlib,shutil,subprocess
from pathlib import Path
PLAN_SHA256='90e52e6443cabe7503fd7420624a09109bb6bb64f2351bb511ad0382deb4fd69'
LOCK_SHA256='8ab9b7ae24fe5907965e6dc7b7e226bdcd7f74a845ccc2bfee995f2fc627d0cf'
HOLDOUT_SHA256='818b491f19541c7703cc311afd498fe1169c223a1c03fb4902e275b46b7a9106'
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working');ROOT=WORK/'work03';POST=WORK/'posthoc14b'
def find(name):return sorted(Path(d)/name for d,_,fs in os.walk(INPUT,followlinks=True) if name in fs)
locks=[p for p in find('frozen.json') if p.parent.name=='outputs' and p.parent.parent.name=='work03']
assert len(locks)==1,locks
assert hashlib.sha256(locks[0].read_bytes()).hexdigest()==LOCK_SHA256,'Mounted train bundle is not the frozen lock'
shutil.copytree(locks[0].parent.parent,ROOT,ignore=shutil.ignore_patterns('__pycache__','trainer'))
POST.mkdir(parents=True,exist_ok=True)
(POST/'plan_sha256.txt').write_text(PLAN_SHA256+'\n',encoding='utf-8')
(WORK/'posthoc14b.py').write_text('"""Post-hoc analysis after the freeze (plan: evaluation/posthoc14b-plan.md). Not part of the frozen system.\n\nRuns Qwen3-14B through the frozen pipeline: load() (4-bit nf4, double quant, fp16 compute, one T4), messages(),\ngenerate() (greedy), audit_one() (one repair) and the output contract are imported unchanged from the frozen bundle.\nOnly the model id and revision that the frozen load() reads from config() are switched to 14B; no adapter is loaded\n(the 8B LoRA does not fit 14B).\nOrder: environment checks (disk, chat template, finite logits on a training prompt) -> hash-check and read the 8 test\narticles -> batch audit. WORK03_ROOT points at the copied frozen bundle; outputs go to POSTHOC_OUT, outside the bundle."""\nimport json\nimport os\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\n\nROOT = Path(os.environ[\'WORK03_ROOT\'])\nOUT = Path(os.environ[\'POSTHOC_OUT\'])\nHOLDOUT = Path(os.environ[\'HOLDOUT_PATH\'])\nHOLDOUT_SHA256 = os.environ[\'HOLDOUT_SHA256\']\nMODEL_ID = \'Qwen/Qwen3-14B\'\nREVISION = \'40c069824f4251a91eefaf281ebe4c544efd3e18\'\nNEEDED_FREE_BYTES = 35 * 10**9  # 29.6 GB of weights plus headroom\nsys.path.insert(0, str(ROOT / \'scripts\'))\nimport workflow as wf  # noqa: E402  frozen bundle code, imported unchanged\n\nFROZEN_CONFIG = wf.config\nFROZEN = FROZEN_CONFIG()\n\n\ndef config_14b():\n    """Frozen run_config.json with only the model id and revision replaced; max_new_tokens and context_limit stay."""\n    return {**FROZEN_CONFIG(), \'model_id\': MODEL_ID, \'model_revision\': REVISION}\n\n\ndef disk(paths):\n    out = {}\n    for p in paths:\n        if p and os.path.exists(p):\n            u = shutil.disk_usage(p)\n            out[p] = dict(total_gb=round(u.total / 1e9, 1), free_gb=round(u.free / 1e9, 1))\n    return out\n\n\ndef pick_cache():\n    """Keep the 29.6 GB Hugging Face cache out of /kaggle/working (saved output, about 20 GB)."""\n    cands = [p for p in [\'/tmp\', \'/root\', \'/kaggle/tmp\'] if os.path.isdir(p)]\n    best = max(cands, key=lambda p: shutil.disk_usage(p).free)\n    return Path(best) / \'hf\', shutil.disk_usage(best).free\n\n\ndef versions():\n    import importlib.metadata as meta\n    import torch\n    return dict(torch=torch.__version__, **{p: meta.version(p) for p in\n                                          [\'transformers\', \'peft\', \'bitsandbytes\', \'accelerate\', \'huggingface_hub\']})\n\n\ndef main():\n    OUT.mkdir(parents=True, exist_ok=True)\n    cache, free = pick_cache()\n    os.environ[\'HF_HOME\'] = str(cache)  # before transformers/huggingface_hub are imported\n    os.environ[\'HF_HUB_CACHE\'] = str(cache / \'hub\')  # a preset HF_HUB_CACHE would otherwise win over HF_HOME\n    env = dict(started_utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()), model_id=MODEL_ID, revision=REVISION,\n               weights=\'4-bit nf4 via frozen load(adapter=False); no adapter\',\n               frozen_model=dict(model_id=FROZEN[\'model_id\'], revision=FROZEN[\'model_revision\']),\n               hf_home=str(cache), disk=disk([\'/\', \'/tmp\', \'/root\', \'/kaggle/tmp\', \'/kaggle/working\']),\n               lock_sha256=wf.sha(ROOT / \'outputs/frozen.json\'))\n    wf.write(OUT / \'env_14b.json\', env)\n    assert free >= NEEDED_FREE_BYTES, f\'Not enough disk for the 14B weights: {free / 1e9:.1f} GB free at {cache.parent}\'\n\n    import torch\n    from huggingface_hub import snapshot_download\n    from transformers import AutoTokenizer\n    env.update(versions=versions(), gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])\n\n    t0 = time.monotonic()\n    snapshot_download(MODEL_ID, revision=REVISION, max_workers=8)\n    env[\'seconds_download\'] = round(time.monotonic() - t0, 1)\n    env[\'disk_after_download\'] = disk([str(cache.parent)])\n\n    # Same chat template and tokenization as the frozen 8B, checked on a training article (test data not read yet).\n    row = wf.data(\'train12\')[0]\n    tok8 = AutoTokenizer.from_pretrained(FROZEN[\'model_id\'], revision=FROZEN[\'model_revision\'])\n    tok14 = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)\n    render = lambda t: t.apply_chat_template(wf.messages(row), tokenize=False, add_generation_prompt=True,\n                                             enable_thinking=False)\n    env[\'chat_template_same_as_8b\'] = tok8.chat_template == tok14.chat_template\n    env[\'train_prompt_tokens_same_as_8b\'] = (tok8(render(tok8), add_special_tokens=False).input_ids\n                                             == tok14(render(tok14), add_special_tokens=False).input_ids)\n    wf.write(OUT / \'env_14b.json\', env)\n\n    wf.config = config_14b  # the frozen load() and generate() read config() at call time\n    t0 = time.monotonic()\n    tok, model = wf.load(adapter=False)\n    env.update(seconds_load=round(time.monotonic() - t0, 1), parameters=model.num_parameters(),\n               footprint_bytes=model.get_memory_footprint(),\n               devices=sorted(set(str(v) for v in model.hf_device_map.values())),\n               quantization_config=json.loads(json.dumps(model.config.quantization_config.to_dict(), default=str)),\n               gpu0_allocated_after_load=torch.cuda.memory_allocated(0))\n\n    # fp16 overflow check: one full training prompt, last-position logits must be finite.\n    x = tok(render(tok), add_special_tokens=False, return_tensors=\'pt\').to(\'cuda:0\')\n    with torch.inference_mode():\n        logits = model(**x, logits_to_keep=1).logits\n    env[\'logits_check\'] = dict(train_id=row[\'ID\'], input_tokens=int(x.input_ids.shape[1]),\n                               finite=bool(torch.isfinite(logits).all()),\n                               top_token=tok.decode([int(logits[0, -1].argmax())]))\n    wf.write(OUT / \'env_14b.json\', env)\n    assert env[\'logits_check\'][\'finite\'], \'Non-finite logits in fp16; stop before reading the test data\'\n    del x, logits\n\n    # Test data: hash-check, then read through the frozen data()/batch().\n    assert wf.sha(HOLDOUT) == HOLDOUT_SHA256, \'holdout8.csv bytes changed\'\n    shutil.copyfile(HOLDOUT, ROOT / \'data/holdout8.csv\')\n    env[\'holdout_sha256\'] = wf.sha(ROOT / \'data/holdout8.csv\')\n    t0 = time.monotonic()\n    wf.batch(wf.data(\'holdout8\'), lambda msg: wf.generate(tok, model, msg), OUT / \'holdout_14b.jsonl\')\n    env.update(seconds_generation=round(time.monotonic() - t0, 1),\n               finished_utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()),\n               note=\'peak_gpu_bytes inside each record is per generate() call on device 0\')\n    wf.write(OUT / \'env_14b.json\', env)\n    print(json.dumps(env, ensure_ascii=False, indent=1))\n\n\nif __name__ == \'__main__\':\n    main()\n',encoding='utf-8')
subprocess.run([sys.executable,'-m','pip','install',*['transformers==4.57.6', 'peft==0.20.0', 'accelerate==1.14.0', 'bitsandbytes==0.50.2', 'datasets==4.8.5', 'jsonschema>=4,<5', 'pandas>=2,<3']],check=True)
env={**os.environ,'WORK03_ROOT':str(ROOT),'POSTHOC_OUT':str(POST),'HOLDOUT_SHA256':HOLDOUT_SHA256,
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
# Frozen check: every locked file (prompt, knowledge, config, data, scripts, adapter) must match the lock.
subprocess.run([sys.executable,str(ROOT/'scripts/workflow.py'),'check-freeze'],check=True,env=env)
for p in ['/','/tmp','/root','/kaggle/working']:
    u=shutil.disk_usage(p);print(p,f'free {u.free/1e9:.1f} GB of {u.total/1e9:.1f} GB')

## Qwen3-14B、4-bit、不載微調，審 8 篇測試文章
先下載固定版本的權重並做環境檢查，通過後才核對測試資料雜湊並讀取。

In [ ]:
# Only locate the test CSV here; posthoc14b.py hash-checks and reads it after the environment checks pass.
holdouts=find('holdout8.csv')
assert len(holdouts)==1,holdouts
subprocess.run([sys.executable,str(WORK/'posthoc14b.py')],check=True,
               env={**env,'HOLDOUT_PATH':str(holdouts[0]),'CUDA_VISIBLE_DEVICES':'0'})

## 結果摘要
逐筆判讀與指標在本機依預先登錄的計畫計算。

In [ ]:
import pandas as pd
from IPython.display import display
rows=[json.loads(s) for s in (POST/'holdout_14b.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
display(pd.DataFrame([dict(ID=r['ID'],first_pass_valid=r['first_pass_valid'],final_valid=r['final_valid'],
  Score=(r['parsed'] or {}).get('Score'),codes=','.join(x['code'] for x in (r['parsed'] or {}).get('Reasons',[])),
  severities=','.join(x['severity'] for x in (r['parsed'] or {}).get('Reasons',[])),
  seconds=round(sum(a['seconds'] for a in r['attempts']),1)) for r in rows]))
print(json.dumps(json.loads((POST/'env_14b.json').read_text(encoding='utf-8')),ensure_ascii=False,indent=1))
# Keep only the post-hoc outputs; the frozen bundle and the test CSV stay in their original kernel and dataset.
shutil.rmtree(ROOT)
print(sorted(p.name for p in POST.iterdir()))